In [66]:
from pathlib import Path
import json
from collections import Counter

# ============================================================
# PROJECT PATHS
# ============================================================

ROOT = Path(
    r"C:\amrita_uni\s6\NLP\project\Rubric-based-evaluation-of-PL-SQL-code\Rubric-based-evaluation-of-PL-SQL-code"
)

SCHEMA_ARTIFACT_DIR = (
    ROOT /
    "artifacts" /
    "schema_output"
)

PLANNING_ARTIFACT_DIR = (
    ROOT /
    "artifacts" /
    "test_planning_output"
)

ARTIFACT_DIR = (
    ROOT /
    "artifacts" /
    "testcase_generation_output"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Paths initialized.")

# ============================================================
# LOAD ARTIFACTS
# ============================================================

schema = json.loads(
    (
        SCHEMA_ARTIFACT_DIR /
        "schema.json"
    ).read_text(
        encoding="utf-8"
    )
)

ddl_text = (
    SCHEMA_ARTIFACT_DIR /
    "ddl.sql"
).read_text(
    encoding="utf-8"
)

operations = json.loads(
    (
        SCHEMA_ARTIFACT_DIR /
        "operations.json"
    ).read_text(
        encoding="utf-8"
    )
)

operation_behavior_map = json.loads(
    (
        SCHEMA_ARTIFACT_DIR /
        "operation_behavior_map.json"
    ).read_text(
        encoding="utf-8"
    )
)

combined_test_plan = json.loads(
    (
        PLANNING_ARTIFACT_DIR /
        "combined_test_plan.json"
    ).read_text(
        encoding="utf-8"
    )
)

print("Artifacts loaded.")

print(
    "\nOperations:",
    len(operations)
)

print(
    "Planned tests:",
    len(combined_test_plan)
)

Paths initialized.
Artifacts loaded.

Operations: 3
Planned tests: 10


In [67]:
# ============================================================
# OPERATION CATALOG
# ============================================================

def build_operation_catalog(
    operations
):

    catalog = {}

    for operation in operations:

        catalog[
            operation[
                "operation_name"
            ]
        ] = operation

    return catalog


operation_catalog = (
    build_operation_catalog(
        operations
    )
)

print(
    json.dumps(
        operation_catalog,
        indent=2
    )
)

{
  "UpdateCustomerClass": {
    "operation_name": "UpdateCustomerClass",
    "operation_type": "function",
    "semantic_categories": [
      "classification_logic"
    ],
    "parameters": [
      {
        "name": "threshold_value",
        "type": "NUMBER"
      },
      {
        "name": "cust_no",
        "type": "NUMBER"
      }
    ],
    "expected_effects": [
      "Updates the c_type of a customer based on their account balance relative to a threshold value."
    ]
  },
  "CloseBranch": {
    "operation_name": "CloseBranch",
    "operation_type": "function",
    "semantic_categories": [
      "transaction_sensitive_operation",
      "cross_table_dependency"
    ],
    "parameters": [
      {
        "name": "closing_branch_no",
        "type": "NUMBER"
      },
      {
        "name": "new_branch_no",
        "type": "NUMBER"
      }
    ],
    "expected_effects": [
      "Transfers all accounts from the closing branch to a new branch, and removes the closing branch."
    ]
 

In [68]:
import hashlib
import subprocess
import re


def clean_cli_output(text: str) -> str:

    text = re.sub(
        r"\x1b\[[0-9;]*m",
        "",
        text
    )

    text = text.replace(
        "```json",
        ""
    )

    text = text.replace(
        "```",
        ""
    )

    return text.strip()



CACHE_DIR = (
    ROOT /
    ".cache" /
    "testcase_generation"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def ollama_chat(
    messages,
    model="qwen2.5:7b"
):

    payload = json.dumps(
        messages,
        sort_keys=True
    )

    cache_key = hashlib.sha256(
        payload.encode()
    ).hexdigest()

    cache_file = (
        CACHE_DIR /
        f"{cache_key}.txt"
    )

    if cache_file.exists():

        return cache_file.read_text(
            encoding="utf-8"
        )

    full_prompt = "\n\n".join(

        f"{m['role'].upper()}:\n{m['content']}"

        for m in messages
    )

    result = subprocess.run(

        [
            "ollama",
            "run",
            model
        ],

        input=full_prompt,

        text=True,

        capture_output=True,

        encoding="utf-8"
    )

    output = clean_cli_output(
        result.stdout
    )

    cache_file.write_text(
        output,
        encoding="utf-8"
    )

    return output

def clean_json_blob(
    text
):

    start = text.find("{")
    end = text.rfind("}")

    if (
        start == -1
        or
        end == -1
    ):
        raise ValueError(
            "No JSON found"
        )

    return text[
        start:end+1
    ]

In [69]:
TESTCASE_SYSTEM_PROMPT = """
You are an Oracle PL/SQL testing expert.

Return STRICT VALID JSON ONLY.

DO NOT return explanations.

DO NOT return markdown.

Every testcase MUST use EXACTLY this schema:

{
  "setup_sql": [
    {
      "sql": "...",
      "expected_result": {}
    }
  ],

  "execution_sql": [
    {
      "sql": "...",
      "expected_result": {}
    }
  ],

  "assertion_sql": [
    {
      "sql": "...",
      "expected_result": {}
    }
  ],

  "expected_result": {}
}

Rules:

1. setup_sql cannot be empty.

2. execution_sql cannot be empty.

3. assertion_sql cannot be empty.

4. Always generate Oracle SQL.

5. Use operation parameters.

6. Use schema entities.

7. Return JSON only.
"""

In [70]:
def generate_testcase_with_llm(

    testcase,

    schema,

    operation
):

    payload = {

        "schema":
            schema,

        "operation":
            operation,

        "test_plan":
            testcase
    }

    messages = [

        {
            "role":
                "system",

            "content":
                TESTCASE_SYSTEM_PROMPT
        },

        {
            "role":
                "user",

            "content":
                json.dumps(
                    payload,
                    indent=2
                )
        }
    ]

    for attempt in range(3):

        response = ollama_chat(messages)

        try:

            cleaned = clean_json_blob(
                response
            )

            parsed = json.loads(
                cleaned
            )

            if (
                len(
                    parsed.get(
                        "execution_sql",
                        []
                    )
                ) > 0
            ):

                return parsed

        except Exception:

            pass

    print(
        "Failed after retries"
    )

    return {
        "setup_sql": [],
        "execution_sql": [],
        "assertion_sql": [],
        "expected_result": {}
    }

In [71]:
# ============================================================
# LLM TESTCASE SYNTHESIS ENGINE
# ============================================================

def synthesize_testcases():

    generated = []

    testcase_counter = 1

    for testcase in combined_test_plan:

        operation_name = (
            testcase[
                "target_operations"
            ][0]
        )

        operation = (
            operation_catalog[
                operation_name
            ]
        )

        print(
            f"Generating TC-{testcase_counter:03d}"
        )

        llm_output = (
            generate_testcase_with_llm(

                testcase,

                schema,

                operation
            )
        )

        generated.append({

            "testcase_id":
                f"TC-{testcase_counter:03d}",

            "testcase_type":
                testcase[
                    "testcase_type"
                ],

            "test_intent":
                testcase[
                    "test_intent"
                ],

            "target_operation":
                operation_name,

            "risk_pattern":
                testcase.get(
                    "risk_pattern"
                ),

            "behavioral_focus":
                testcase.get(
                    "behavioral_focus",
                    []
                ),

            "setup_sql":
                llm_output.get(
                    "setup_sql",
                    []
                ),

            "execution_sql":
                llm_output.get(
                    "execution_sql",
                    []
                ),

            "assertion_sql":
                llm_output.get(
                    "assertion_sql",
                    []
                ),

            "expected_result":
                llm_output.get(
                    "expected_result",
                    {}
                )
        })

        testcase_counter += 1

    return generated


generated_testcases = (
    synthesize_testcases()
)

print(
    "\nGenerated:",
    len(
        generated_testcases
    )
)

Generating TC-001
Generating TC-002
Generating TC-003
Failed after retries
Generating TC-004
Generating TC-005
Generating TC-006
Generating TC-007
Failed after retries
Generating TC-008
Generating TC-009
Generating TC-010

Generated: 10


In [72]:
print(
    json.dumps(
        generated_testcases[0],
        indent=2
    )
)

{
  "testcase_id": "TC-001",
  "testcase_type": "edge_case",
  "test_intent": "exact_balance_validation",
  "target_operation": "SafeWithdrawal",
  "risk_pattern": "rollback_failure",
  "behavioral_focus": [
    "transaction_consistency_test",
    "rollback_validation"
  ],
  "setup_sql": [
    {
      "sql": "INSERT INTO ACCOUNTS (ac_no, br_no, cust_no, ac_type, bal) VALUES (101, 10, 201, 'SAVINGS', 100)",
      "expected_result": {}
    },
    {
      "sql": "INSERT INTO BRANCHES (br_no, br_name, loc) VALUES (10, 'Main Branch', 'New York')",
      "expected_result": {}
    },
    {
      "sql": "INSERT INTO CUSTOMER (cno, cname, c_type) VALUES (201, 'John Doe', NULL)",
      "expected_result": {}
    }
  ],
  "execution_sql": [
    {
      "sql": "BEGIN SafeWithdrawal(101, 100); END;",
      "expected_result": {}
    }
  ],
  "assertion_sql": [
    {
      "sql": "SELECT bal FROM ACCOUNTS WHERE ac_no = 101",
      "expected_result": {
        "columns": [
          "bal"
        ],
 

In [73]:
# ============================================================
# EXECUTION BUNDLE
# ============================================================

def build_execution_bundle(
    ddl_text: str,
    generated_testcases: list
):

    return {

        "schema_ddl":
            ddl_text,

        "testcase_count":
            len(
                generated_testcases
            ),

        "testcases":
            generated_testcases
    }


execution_bundle = (
    build_execution_bundle(

        ddl_text,

        generated_testcases
    )
)

print(
    json.dumps(
        {

            "testcase_count":
                execution_bundle[
                    "testcase_count"
                ]

        },

        indent=2
    )
)

{
  "testcase_count": 10
}


In [74]:
# ============================================================
# TESTCASE METADATA
# ============================================================

def build_metadata(
    testcases
):

    operation_distribution = Counter()

    testcase_types = Counter()

    total_setup_sql = 0

    total_execution_sql = 0

    total_assertions = 0

    for testcase in testcases:

        testcase_types[
            testcase[
                "testcase_type"
            ]
        ] += 1

        operation_distribution[
            testcase[
                "target_operation"
            ]
        ] += 1

        total_setup_sql += len(
            testcase[
                "setup_sql"
            ]
        )

        total_execution_sql += len(
            testcase[
                "execution_sql"
            ]
        )

        total_assertions += len(
            testcase[
                "assertion_sql"
            ]
        )

    return {

        "total_testcases":
            len(testcases),

        "operation_distribution":
            dict(
                operation_distribution
            ),

        "testcase_types":
            dict(
                testcase_types
            ),

        "total_setup_sql":
            total_setup_sql,

        "total_execution_sql":
            total_execution_sql,

        "total_assertions":
            total_assertions
    }


testcase_metadata = (
    build_metadata(
        generated_testcases
    )
)

print(
    json.dumps(
        testcase_metadata,
        indent=2
    )
)

{
  "total_testcases": 10,
  "operation_distribution": {
    "SafeWithdrawal": 4,
    "UpdateCustomerClass": 4,
    "CloseBranch": 2
  },
  "testcase_types": {
    "edge_case": 7,
    "normal_case": 3
  },
  "total_setup_sql": 22,
  "total_execution_sql": 8,
  "total_assertions": 9
}


In [75]:
# ============================================================
# EXPORT ARTIFACTS
# ============================================================

(
    ARTIFACT_DIR /
    "test_cases.json"
).write_text(

    json.dumps(
        generated_testcases,
        indent=2
    ),

    encoding="utf-8"
)

(
    ARTIFACT_DIR /
    "execution_bundle.json"
).write_text(

    json.dumps(
        execution_bundle,
        indent=2
    ),

    encoding="utf-8"
)

(
    ARTIFACT_DIR /
    "testcase_metadata.json"
).write_text(

    json.dumps(
        testcase_metadata,
        indent=2
    ),

    encoding="utf-8"
)

print(
    "Artifacts exported."
)

Artifacts exported.


In [76]:
# ============================================================
# VALIDATION
# ============================================================

assert (
    len(
        generated_testcases
    )
    ==
    len(
        combined_test_plan
    )
)

assert all(

    len(
        testcase[
            "setup_sql"
        ]
    ) > 0

    for testcase in generated_testcases
)

assert all(

    len(
        testcase[
            "execution_sql"
        ]
    ) > 0

    for testcase in generated_testcases
)

assert all(

    len(
        testcase[
            "assertion_sql"
        ]
    ) > 0

    for testcase in generated_testcases
)

assert (
    execution_bundle[
        "testcase_count"
    ]
    ==
    len(
        generated_testcases
    )
)

print(
    "Notebook 4 validation passed."
)

AssertionError: 

the thing is currently the note 4 is generating the sql codes good but not 100% accurate and this is due to we putting whole burden alone in single prompt to generate whole test case code, so the solution would be to generate a test case skeleton and then generate by giving it as context to llm. Generate logical testcases and then use them to generate working test cases to reduce halucination